In [ ]:
import os
from google.colab import drive

# CONFIGURATION
# Public clone URL; Git handles any configured credentials.
REPO_OWNER = "theflyingkhui04"
REPO_NAME = "do-an-ptit-d22"
TARGET_DIR = f"/root/{REPO_NAME}"
CLONE_URL = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"

VSCODE_DRIVE_BACKUP = "/content/drive/MyDrive/vllm_colab_backup"


# STEP 1: CONNECT GOOGLE DRIVE
print("\n" + "="*60)
print("==> STEP 1/4: CONNECT GOOGLE DRIVE")
print("="*60)
drive.mount('/content/drive', force_remount=False)
os.makedirs(f"{VSCODE_DRIVE_BACKUP}/.vscode-cli", exist_ok=True)


# STEP 2: SYNC REPOSITORY SOURCE CODE
print("\n" + "="*60)
print("==> STEP 2/4: SYNC REPO SOURCE CODE FROM GITHUB")
print("="*60)
if not os.path.exists(TARGET_DIR):
    !git clone {CLONE_URL} {TARGET_DIR}
else:
    !git -C {TARGET_DIR} pull

!chmod +x {TARGET_DIR}/scripts/*.sh


# STEP 3: SETUP vLLM ENVIRONMENT
print("\n" + "="*60)
print("==> STEP 3/4: INSTALL vLLM ON COLAB SSD")
print("="*60)
!pip install -r {TARGET_DIR}/requirements-server.txt

print("\n--> Removing torchaudio to prevent CUDA conflict...")
!pip uninstall -y torchaudio



# STEP 4: SETUP VS CODE CLI & LAUNCH TUNNEL
print("\n" + "="*60)
print("==> STEP 4/4: INITIALIZE VS CODE TUNNEL 'colab-t4'")
print("="*60)

# 1. Download CLI if missing (Use Microsoft Direct Link to avoid DNS/Redirect issues on T4 GPU)
if os.path.exists("./code"):
    print("--> Found existing VS Code CLI, removing for fresh install...")
    !rm -rf ./code
if os.path.exists("vscode_cli.tar.gz"):
    print("--> Found existing vscode_cli.tar.gz, removing for fresh download...")
    !rm -f vscode_cli.tar.gz

print("--> Downloading and extracting VS Code CLI (Direct Link)...")
!wget -O vscode_cli.tar.gz "https://update.code.visualstudio.com/latest/cli-alpine-x64/stable" && tar -xf vscode_cli.tar.gz

# 2. Link credential storage directory to Drive
os.makedirs(f"{VSCODE_DRIVE_BACKUP}/.vscode-cli", exist_ok=True)
!ln -sfn {VSCODE_DRIVE_BACKUP}/.vscode-cli /root/.vscode-cli

# 3. Log in with GitHub provider
print("--> Checking GitHub login:")
!./code tunnel user login --provider github

# 4. Start Tunnel
print("\n" + "="*60)
print("  LAUNCHING TUNNEL 'colab-t4'...")
print("="*60 + "\n")
!./code tunnel --accept-server-license-terms --name colab-t4